# MedRAG — Train Retriever trên KAGGLE (100k cặp, 2 epoch, hard negative + AMP)

Notebook cho **Kaggle** (không phải Colab). Khác biệt chính:
- Không có Google Drive. Data đọc từ `/kaggle/input/...`, kết quả ghi vào `/kaggle/working/`.
- Checkpoint + model cuối lưu trong `/kaggle/working/` (tải về sau khi xong).

## CHUẨN BỊ TRƯỚC KHI CHẠY (làm 1 lần)
1. **Upload data thành Kaggle Dataset:**
   - Vào https://www.kaggle.com/datasets → **New Dataset**.
   - Kéo file `train_pairs.jsonl` (bản đầy đủ 211k, ở máy local) vào, đặt tên ví dụ `medrag-pairs`, bấm **Create**.
2. **Trong notebook này:** panel bên phải → **Add Input** → tìm dataset `medrag-pairs` vừa tạo → Add.
   File sẽ nằm ở `/kaggle/input/medrag-pairs/train_pairs.jsonl`.
3. **Bật GPU:** panel phải → **Settings** (hoặc ⋮) → **Accelerator** → chọn **GPU T4 x2** (hoặc P100).
4. **Bật Internet:** Settings → **Internet** → **On** (cần để git clone + tải BioBERT). 
   Kaggle yêu cầu xác minh số điện thoại để bật Internet — làm theo hướng dẫn nếu được nhắc.
5. Chạy tuần tự từ Cell 1.


In [1]:
# Cell 1 — Kiểm tra GPU + tìm file dữ liệu trong /kaggle/input
import os, glob, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "KHÔNG THẤY GPU — bật Accelerator!")

hits = glob.glob("/kaggle/input/**/train_pairs.jsonl", recursive=True)
print("Tìm thấy train_pairs.jsonl tại:", hits)
assert hits, "Chưa Add dataset chứa train_pairs.jsonl vào notebook (panel phải -> Add Input)."
SRC_PAIRS = hits[0]
!wc -l "$SRC_PAIRS"


Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

Tìm thấy train_pairs.jsonl tại: ['/kaggle/input/datasets/mannual2233/medrag-pairs/train_pairs.jsonl']
211269 /kaggle/input/datasets/mannual2233/medrag-pairs/train_pairs.jsonl


In [2]:
# Cell 2 — Cài code repo (Internet phải đang BẬT)
!git clone -q https://github.com/muahahaha55/Medical-RAG-retriever.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -e . 2>/dev/null
!pip install -q sentence-transformers==2.7.0 2>/dev/null
print("Đã cài xong.")


/kaggle/working/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for medrag-retriever (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 96.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.7 MB/s eta 0:00:00
Đã cài xong.


In [3]:
# Cell 3 — SUBSAMPLE: giữ TOÀN BỘ hard negative + sample random cho đủ TARGET
import json, random

TARGET_PAIRS = 100000   # <<< đổi ở đây: 50000 / 100000 / 150000. Kaggle 9h+ nên 100k rất an toàn.
SEED = 42

hard, rand = [], []
with open(SRC_PAIRS) as f:
    for line in f:
        r = json.loads(line)
        (hard if r.get("negative_type") == "hard" else rand).append(r)
print(f"Nguồn: {len(hard)} hard + {len(rand)} random = {len(hard)+len(rand)}")

need_random = max(0, TARGET_PAIRS - len(hard))
random.seed(SEED)
subset = hard + random.sample(rand, min(need_random, len(rand)))
random.shuffle(subset)

os.makedirs("/kaggle/working/data", exist_ok=True)
OUT_PAIRS = "/kaggle/working/data/train_pairs_sub.jsonl"
with open(OUT_PAIRS, "w") as f:
    for r in subset:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

n_hard = sum(1 for r in subset if r.get("negative_type") == "hard")
print(f"Subset: {len(subset)} cặp ({n_hard} hard giữ 100%, {len(subset)-n_hard} random) -> {OUT_PAIRS}")


Nguồn: 20000 hard + 191269 random = 211269
Subset: 100000 cặp (20000 hard giữ 100%, 80000 random) -> /kaggle/working/data/train_pairs_sub.jsonl


In [4]:
# Cell 4 — CẤU HÌNH
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

WORK_CKPT  = "/kaggle/working/checkpoints"          # checkpoint (an toàn trong session)
WORK_FINAL = "/kaggle/working/MedicalRetriever-v1"  # model cuối -> tải về sau
os.makedirs(WORK_CKPT, exist_ok=True)

EPOCHS      = 2
BATCH_SIZE  = 32          # AMP + T4 chịu được 32; nếu OOM hạ 24 -> 16
MAX_SEQ_LEN = 256
LR          = 2e-5
CKPT_EVERY  = 500
USE_AMP     = True
print(f"EPOCHS={EPOCHS} | BATCH={BATCH_SIZE} | SEQ={MAX_SEQ_LEN} | AMP={USE_AMP}")


EPOCHS=2 | BATCH=32 | SEQ=256 | AMP=True


In [6]:
# Cell 5 — ƯỚC LƯỢNG THỜI GIAN TỔNG (không phải 1 epoch)
n = sum(1 for _ in open(OUT_PAIRS))
spe = -(-n // BATCH_SIZE)
total = spe * EPOCHS
print(f"{n} cặp | {spe} step/epoch | {total} step cho TỔNG {EPOCHS} epoch")
for s in (1.0, 1.3):
    print(f"  ~{s}s/step -> tổng ~ {total*s/3600:.1f} giờ")
print(f"\n⚠️ tqdm chỉ hiện ETA 1 EPOCH. Thời gian THẬT = ETA × {EPOCHS}.")


100000 cặp | 3125 step/epoch | 6250 step cho TỔNG 2 epoch
  ~1.0s/step -> tổng ~ 1.7 giờ
  ~1.3s/step -> tổng ~ 2.3 giờ

⚠️ tqdm chỉ hiện ETA 1 EPOCH. Thời gian THẬT = ETA × 2.


In [5]:
# Cell 6 — TRAIN (checkpoint + tự resume trong session + AMP)
import glob
from sentence_transformers import SentenceTransformer, InputExample, losses, models
from torch.utils.data import DataLoader

def latest_ckpt(root):
    subs = [d for d in glob.glob(os.path.join(root,"*")) if os.path.basename(d).isdigit()]
    if not subs: return None, 0
    subs.sort(key=lambda p:int(os.path.basename(p)))
    return subs[-1], int(os.path.basename(subs[-1]))

resume_dir, resume_step = latest_ckpt(WORK_CKPT)
if resume_dir:
    print(f"➜ RESUME từ checkpoint step {resume_step}: {resume_dir}")
    model = SentenceTransformer(resume_dir); model.max_seq_length = MAX_SEQ_LEN
else:
    print("➜ Bắt đầu MỚI từ BioBERT")
    we = models.Transformer("dmis-lab/biobert-base-cased-v1.2", max_seq_length=MAX_SEQ_LEN)
    pool = models.Pooling(we.get_word_embedding_dimension(), pooling_mode="mean")
    model = SentenceTransformer(modules=[we, pool])

examples = []
with open(OUT_PAIRS) as f:
    for line in f:
        r = json.loads(line)
        texts = [r["query"], r["positive"]]
        if r.get("negative"): texts.append(r["negative"])
        examples.append(InputExample(texts=texts))
print(f"{len(examples)} InputExample")
loader = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
loss = losses.MultipleNegativesRankingLoss(model)

warmup = 0 if resume_dir else int(len(loader) * EPOCHS * 0.1)
print("warmup_steps =", warmup)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS, warmup_steps=warmup,
    optimizer_params={"lr": LR},
    output_path=WORK_FINAL,
    checkpoint_path=WORK_CKPT,
    checkpoint_save_steps=CKPT_EVERY,
    checkpoint_save_total_limit=3,
    use_amp=USE_AMP, show_progress_bar=True,
)
print("✅ XONG. Model cuối tại", WORK_FINAL)


➜ Bắt đầu MỚI từ BioBERT


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/sentence_transformers/SentenceTransformer.py:1011: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch:   0%|          | 0/2 [00:00<?, ?it/s]

Iteration:   0%|          | 0/3125 [00:00<?, ?it/s]

Iteration:   0%|          | 0/3125 [00:00<?, ?it/s]

✅ XONG. Model cuối tại /kaggle/working/MedicalRetriever-v1


In [6]:
# Cell 7 — Kiểm tra + hướng dẫn tải model về
!ls -la /kaggle/working/MedicalRetriever-v1/
print()
print("TẢI VỀ: panel phải -> Output -> mở thư mục MedicalRetriever-v1 -> tải từng file,")
print("HOẶC bấm 'Save Version' (Quick Save) để Kaggle lưu toàn bộ /kaggle/working làm Output,")
print("sau đó tải cả bộ hoặc dùng lại ở notebook khác qua 'Add Input'.")


total 424024
drwxr-xr-x 4 root root      4096 Jul  4 06:17 .
drwxr-xr-x 7 root root      4096 Jul  4 05:32 ..
drwxr-xr-x 2 root root      4096 Jul  4 06:17 1_Pooling
-rw-r--r-- 1 root root       599 Jul  4 06:17 config.json
-rw-r--r-- 1 root root       172 Jul  4 06:17 config_sentence_transformers.json
drwxr-xr-x 2 root root      4096 Jul  4 05:32 eval
-rw-r--r-- 1 root root 433263448 Jul  4 06:17 model.safetensors
-rw-r--r-- 1 root root       229 Jul  4 06:17 modules.json
-rw-r--r-- 1 root root      3926 Jul  4 06:17 README.md
-rw-r--r-- 1 root root        53 Jul  4 06:17 sentence_bert_config.json
-rw-r--r-- 1 root root       125 Jul  4 06:17 special_tokens_map.json
-rw-r--r-- 1 root root      1300 Jul  4 06:17 tokenizer_config.json
-rw-r--r-- 1 root root    669175 Jul  4 06:17 tokenizer.json
-rw-r--r-- 1 root root    213450 Jul  4 06:17 vocab.txt

TẢI VỀ: panel phải -> Output -> mở thư mục MedicalRetriever-v1 -> tải từng file,
HOẶC bấm 'Save Version' (Quick Save) để Kaggle lưu toàn b